In [1]:
uri = 'https://api.football-data.org/v4/competitions/WC/matches'
headers = {'X-Auth-Token': 'b006736168e34387975ae15e83b341a4'}

In [3]:
import requests
import json
import sys
import os
import streamlit as st
import numpy as np
from names_eng_to_nor import ENGLISH_TO_NORWEGIAN

In [4]:
response = requests.get(uri, headers=headers, timeout=10)
response.raise_for_status()

In [5]:
np.sum([1,2,3,4,5])

np.int64(15)

In [ ]:
matches = []
knockout_stages = {
    'round_of_32': [],
    'round_of_16': [],
    'quarter_finals': [],
    'semi_finals': [],
    'finals_teams': [],
    'finals_winner': None
}

# Stage mapping from API to our format
stage_mapping = {
    'LAST_32': 'round_of_32',
    'LAST_16': 'round_of_16',
    'QUARTER_FINALS': 'quarter_finals',
    'SEMI_FINALS': 'semi_finals',
    'FINAL': 'finals'
}

# Parse JSON response
for match in response.json()['matches']:
    
    home_team = match['homeTeam']['name']
    away_team = match['awayTeam']['name']
    stage = match.get('stage', 'GROUP_STAGE')
    status = match.get('status')
    
    # Convert team names to Norwegian
    home_team_nor = ENGLISH_TO_NORWEGIAN.get(home_team, home_team)
    away_team_nor = ENGLISH_TO_NORWEGIAN.get(away_team, away_team)
    
    # Process matches: Include all Group Stage matches (FINISHED, IN_PLAY, TIMED)
    # and only FINISHED matches for knockout stages to track winners
    if status == 'FINISHED' or (stage == 'GROUP_STAGE' and status in ['IN_PLAY', 'TIMED']):
        home_score = match['score']['fullTime']['home']
        away_score = match['score']['fullTime']['away']
        
        match_result = {
            'home_team_eng': home_team,
            'away_team_eng': away_team,
            'home_team': home_team_nor,
            'away_team': away_team_nor,
            'home_score': home_score,
            'away_score': away_score,
            'score_str': f"{home_score}–{away_score}" if home_score is not None else "–"
        }
        
        # Add to appropriate list
        if stage == 'GROUP_STAGE':
            matches.append(match_result)
        elif status == 'FINISHED' and stage in stage_mapping:
            # Determine winner for knockout stages
            if home_score > away_score:
                winner = home_team_nor
            elif away_score > home_score:
                winner = away_team_nor
            else:
                # For knockout stages, there shouldn't be draws (goes to extra time/penalties)
                # But if it happens, we'll skip it
                continue
            
            if stage == 'FINAL':
                knockout_stages['finals_winner'] = winner
                # Also add both finalists to finals_teams
                if home_team_nor not in knockout_stages['finals_teams']:
                    knockout_stages['finals_teams'].append(home_team_nor)
                if away_team_nor not in knockout_stages['finals_teams']:
                    knockout_stages['finals_teams'].append(away_team_nor)
            else:
                stage_key = stage_mapping[stage]
                knockout_stages[stage_key].append(winner)

In [ ]:
print("Matches fetched and processed successfully.")

In [8]:
display(knockout_stages)

{'round_of_32': [],
 'round_of_16': [],
 'quarter_finals': [],
 'semi_finals': [],
 'finals_teams': [],
 'finals_winner': None}

In [16]:
print(response.json()['matches'][73]['stage'])

LAST_32


In [18]:
response.json()['matches'][74]

{'area': {'id': 2267, 'name': 'World', 'code': 'INT', 'flag': None},
 'competition': {'id': 2000,
  'name': 'FIFA World Cup',
  'code': 'WC',
  'type': 'CUP',
  'emblem': 'https://crests.football-data.org/wm26.png'},
 'season': {'id': 2398,
  'startDate': '2026-06-11',
  'endDate': '2026-07-19',
  'currentMatchday': 2,
  'winner': None},
 'id': 537415,
 'utcDate': '2026-06-29T20:30:00Z',
 'status': 'TIMED',
 'matchday': None,
 'stage': 'LAST_32',
 'group': None,
 'lastUpdated': '2026-06-22T18:05:47Z',
 'homeTeam': {'id': 759,
  'name': 'Germany',
  'shortName': 'Germany',
  'tla': 'GER',
  'crest': 'https://crests.football-data.org/759.svg'},
 'awayTeam': {'id': None,
  'name': None,
  'shortName': None,
  'tla': None,
  'crest': None},
 'score': {'winner': None,
  'duration': 'REGULAR',
  'fullTime': {'home': None, 'away': None},
  'halfTime': {'home': None, 'away': None}},
 'odds': {'msg': 'Activate Odds-Package in User-Panel to retrieve odds.'},
 'referees': []}